# Evaluación Parcial N°1: Clasificación de Enfermedades en Hojas de Tomate (PlantVillage)

**Contexto del Problema:**
Las enfermedades en los cultivos de tomate generan pérdidas masivas. El objetivo de este notebook es implementar un Perceptrón Multicapa (MLP) capaz de clasificar si una hoja está sana o tiene alguna enfermedad específica.




**README y github:** https://github.com/AngeLDurand/Plant_Village/tree/main



In [2]:
!mkdir -p ~/.kaggle && echo "KGAT_34f54de15b64347047e43ecc5dd9519b" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [3]:
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d dataset
print("¡Dataset descargado y descomprimido exitosamente!")

Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:04<00:00, 153MB/s]

¡Dataset descargado y descomprimido exitosamente!


# Carga, seleccion y transformación.

Usamos image_dataset_from_directory con la funcion keras para traer las imagenes directo desde las carpetas sin saturar la memoria del computador.

Nos sirve para ordenar las fotos en pequeños grupos (lotes) y filtrar de inmediato solo las 10 categorias de tomate que necesitamos, dejando fuera el resto del dataset.

In [26]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Definimos el tamaño de la imagen y el tamaño del lote (batch)
tamano_imagen = (128, 128) # Un tamaño prudente para MLP
tamano_batch = 32

# Ruta dataset
ruta_dataset = "/content/dataset/PlantVillage"

# Lista con los nombres exactos de las 10 carpetas de tomate en el dataset
clases_tomate = ['Tomato_Bacterial_spot',
                 'Tomato_Early_blight',
                 'Tomato_Late_blight',
                 'Tomato_Leaf_Mold',
                 'Tomato_Septoria_leaf_spot',
                 'Tomato_Spider_mites_Two_spotted_spider_mite',
                 'Tomato__Target_Spot',
                 'Tomato__Tomato_YellowLeaf__Curl_Virus',
                 'Tomato__Tomato_mosaic_virus',
                 'Tomato_healthy'
]

Transformacion: Debido a que la red neuronal solo entiende numeros y no palabras o nombres de las carpetas configuramos label_mode='categorical' para aplicar One-Hot Encoding a las etiquetas, esto convierte las etiquetas en un formato de vectores con ceros y unos

In [33]:
# Cargar el conjunto de entrenamiento (80%)
datos_entrenamiento = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,     # separamos el 20% para validacion
    subset = "training",      # indicamos que este es el conjunto de entrenamiento
    seed=123,                 # semilla para que la division sea siempre igual
    image_size=tamano_imagen,
    batch_size=tamano_batch,
    label_mode="categorical", # etiqueta en formato one-hot necesario
    class_names=clases_tomate #filtramos solo las carpetas de tomate

)

datos_validacion = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    validation_split=0.2,
    subset="validation",     # Indicamos que este es el conjunto de validacion
    seed=123,
    image_size=tamano_imagen,
    batch_size=tamano_batch,
    label_mode="categorical",
    class_names=clases_tomate
)

# guardar los nombres de las clases para usarlos despues

nombres_clases = datos_entrenamiento.class_names
print("Clases cargadas:", nombres_clases)

Found 16011 files belonging to 10 classes.
Using 12809 files for training.
Found 16011 files belonging to 10 classes.
Using 3202 files for validation.
Clases cargadas: ['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


# Transformación, Normalización y Separación(Data Pipeline)

Escalado / Normalizacion (Rescaling): Los pixeles de las fotos vienen en un rango de 0 a 255, lo que resulta en numeros muy grandes que dificultan el aprendizaje de la red. Con esta capa transformamos esos valores para que queden entre 0 y 1, logrando que el modelo entrene de forma mas estable y rapida.

In [34]:
from tensorflow import keras
from tensorflow.keras import layers

# Iniciamos el modelo secuencial
modelo = keras.Sequential()

# 1. Capa de Normalización (Rescaling): píxeles de 0-255 a 0-1
modelo.add(layers.Rescaling(1./255, input_shape=(128, 128, 3)))

Preparacion(Flatten): Las imagenes tienen una forma cuadrada (alto, ancho y canales de color), pero el perceptron multicapa que estamos armando es una red sencilla que solo acepta una fila larga de datos. Esta capa se encarga de estirar toda la estructura de la foto en una sola linea de numeros para que la neurona pueda procesarla.

In [35]:
# 2. Capa Flatten: Aplana la imagen a un vector 1D (requisito del MLP)
modelo.add(layers.Flatten())

Por qué usar ReLU y descartar Leaky ReLU: Leaky ReLU es una gran alternativa (permite un pequeño flujo de valores negativos para evitar que una neurona "muera" o deje de aprender), pero ReLU estándar es más simple, limpia y más que suficiente para un modelo básico de clases de tomate.

Por qué elegimos Softmax en la capa de salida: Es obligatoria para este problema porque tenemos 10 clases y queremos que el modelo nos entregue los resultados en forma de porcentajes de probabilidad (ej: 80% de probabilidad de mancha bacteriana, 10% de tizón temprano, etc., sumando un 100% entre todas).

In [36]:
# 3. Capas Ocultas (Densas) del Perceptrón Multicapa
modelo.add(layers.Dense(256, activation='relu'))
modelo.add(layers.Dense(128, activation='relu'))

# 4. Capa de Salida: 10 neuronas (por las 10 clases), activación softmax
modelo.add(layers.Dense(10, activation='softmax'))

# Ver la estructura final de la red
modelo.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_4 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 49152)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │    12,583,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,617,354 (48.13 MB)

 Trainable params: 12,617,354 (48.13 MB)

 Non-trainable params: 0 (0.00 B)

# Compilacion del modelo

In [37]:
# Fase 3: Compilación del modelo
modelo.compile(
    optimizer='adam',                  # Optimizador eficiente para redes neuronales
    loss='categorical_crossentropy',   # Función de pérdida obligatoria para clasificación multiclase con One-Hot Encoding
    metrics=['accuracy']               # Evaluaremos qué tan exacta es la red
)

print("¡Modelo compilado y listo para entrenar!")

¡Modelo compilado y listo para entrenar!


# Entrenamiento del modelo

In [38]:
# Entrenamos el modelo usando los bloques de datos que ya armamos
history = modelo.fit(
    datos_entrenamiento,
    validation_data=datos_validacion,
    epochs=10 # Puedes probar con 5 o 10 épocas para empezar
)

Epoch 1/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.4112 - loss: 3.1460 - val_accuracy: 0.3888 - val_loss: 1.9854
Epoch 2/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.5181 - loss: 1.4786 - val_accuracy: 0.5834 - val_loss: 1.3306
Epoch 3/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 21s 29ms/step - accuracy: 0.5740 - loss: 1.2732 - val_accuracy: 0.5350 - val_loss: 1.4403
Epoch 4/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.6234 - loss: 1.1015 - val_accuracy: 0.6352 - val_loss: 1.2131
Epoch 5/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.6656 - loss: 0.9882 - val_accuracy: 0.6140 - val_loss: 1.1410
Epoch 6/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.6692 - loss: 0.9537 - val_accuracy: 0.5000 - val_loss: 1.6283
Epoch 7/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.6992 - loss: 0.8896 - val_accuracy: 0.6974 - val_loss: 0.8689
Epoch 8/10
401/401 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step - accuracy: 0.7137 - loss: 0.8314 - 